# R Fundamentals for Bioinformatics

**Tier 0 -- Computational Foundations | Module 5**

---

## Why R for Bioinformatics?

While this course primarily uses Python, R is indispensable in bioinformatics. The **Bioconductor** ecosystem provides over 2,000 packages for genomics, transcriptomics, and proteomics. Tools like **DESeq2** and **edgeR** for differential expression, **Seurat** for single-cell analysis, and **ggplot2** for publication-quality figures are written in R.

This module gives you a working foundation in R so you can:
- Read and understand R code in published papers and pipelines
- Perform basic data manipulation and statistical analysis
- Use Bioconductor packages when Python alternatives are insufficient
- Create R-based visualizations

### How to Run This Notebook

This notebook uses the **R kernel** (IRkernel). You can run it in:
- **Jupyter** with IRkernel installed (`install.packages('IRkernel'); IRkernel::installspec()`)
- **RStudio** by copying code cells into an R script or R Markdown document
- **Google Colab** by changing runtime to R

---

### Learning Objectives

By the end of this module you will be able to:
- Create and manipulate vectors, matrices, and data frames
- Use vectorized operations (R's greatest strength)
- Filter and subset biological data
- Work with R's statistical distribution functions
- Create informative plots
- Read/write tabular data files
- Understand the basics of Bioconductor

## Why this notebook matters

R is the dominant language for statistical analysis in bioinformatics. DESeq2, edgeR, limma, Seurat, and most Bioconductor packages are written in R. Even if you primarily use Python, you will encounter R scripts in published analyses, and many cutting-edge methods are only available in R. This notebook gives you the R fluency needed to run, understand, and adapt these packages — and to write your own analysis scripts.


## How to use this notebook
1. All code cells use the R kernel — make sure your Jupyter environment has IRkernel installed.
2. R's vectorized operations behave differently from Python loops. Pay attention to how operations apply automatically to every element of a vector.
3. When you see unfamiliar syntax, try `?function_name` in an R cell to read the documentation.
4. The bioinformatics examples throughout are intentionally realistic — the same patterns appear in real DESeq2 and Seurat workflows.


## Common stumbling points

- **R indexing starts at 1, not 0**: `x[1]` is the first element. `x[-1]` does NOT mean the last element — it means "everything except the first."
- **`<-` vs `=` for assignment**: Both work, but `<-` is the R convention. Inside function arguments, `=` has a different meaning (keyword argument), so mixing them up causes subtle bugs.
- **Factors look like strings but are not**: A factor stores an integer code plus a lookup table of labels. Functions that expect strings will misbehave if given a factor.
- **Data frames vs. matrices**: Many Bioconductor functions require specific input types. A data frame with character columns cannot be used as a numeric matrix — you must convert explicitly.
- **`library()` vs `require()`**: Use `library()` in scripts; it stops with an error if the package is missing. `require()` returns `FALSE` silently, hiding the problem.


---

## 1. Assignment, Variables, and the Vector Type

Everything in R is an object. The most fundamental object is the **vector** — even a single number is a vector of length 1. This is different from Python, where `x = 5` creates a scalar integer.

R uses `<-` for assignment (reading it as "gets") though `=` also works. The convention in R packages and published code is `<-`, so we follow that here.

### Key differences from Python

| Feature | Python | R |
|---------|--------|---|
| Indexing | Starts at 0 | Starts at **1** |
| Assignment | `=` | `<-` (preferred) or `=` |
| Negative index | `x[-1]` = last element | `x[-1]` = **all except first** |
| Boolean values | `True` / `False` | `TRUE` / `FALSE` |
| Missing data | `None` | `NA` |
| Auto-print | `print(x)` | Just type `x` |


In [ ]:
# Assignment
gene_name <- "TP53"
expression_level <- 5.7
is_oncogene <- FALSE

# Check types
cat("gene_name:", class(gene_name), "\n")
cat("expression_level:", class(expression_level), "\n")
cat("is_oncogene:", class(is_oncogene), "\n")

In [ ]:
# Creating vectors with c() -- "combine"
gene_expression <- c(2.5, 3.1, 4.2, 1.8, 5.6)
gene_names <- c("BRCA1", "TP53", "EGFR", "MYC", "KRAS")
is_oncogene <- c(FALSE, FALSE, TRUE, TRUE, TRUE)

# Vectors have a single type -- mixing types causes coercion
mixed <- c(1, "two", TRUE)
cat("Mixed vector:", mixed, "\n")  # Everything becomes character
cat("Type:", class(mixed), "\n")

In [ ]:
# Sequences -- extremely common in R
positions <- 1:10              # Integer sequence 1 to 10
cat("1:10 =", positions, "\n")

# seq() for more control
coverage <- seq(from = 0, to = 100, by = 10)
cat("seq(0, 100, 10) =", coverage, "\n")

# Evenly spaced values (useful for plotting)
gc_range <- seq(from = 0.3, to = 0.7, length.out = 5)
cat("GC content range:", gc_range, "\n")

# rep() to repeat values
groups <- rep(c("control", "treatment"), each = 3)
cat("Groups:", groups, "\n")

---

## 2. Vectorized Operations -- R's Superpower

In R, arithmetic and logical operations work element-by-element on entire vectors. This is both elegant and fast -- **you rarely need explicit loops in R**.

In [ ]:
x <- 1:5
y <- 6:10

cat("x =", x, "\n")
cat("y =", y, "\n\n")

# Element-wise arithmetic
cat("x + y  =", x + y, "\n")   # 7 9 11 13 15
cat("x * 2  =", x * 2, "\n")   # 2 4 6 8 10
cat("x * y  =", x * y, "\n")   # 6 14 24 36 50
cat("x^2    =", x^2, "\n")     # 1 4 9 16 25

# Common bioinformatics transformations
counts <- c(100, 500, 10, 2000, 50)
cat("\nRaw counts:  ", counts, "\n")
cat("Log2 counts: ", round(log2(counts), 2), "\n")
cat("Log2(x+1):   ", round(log2(counts + 1), 2), "\n")  # Pseudocount to avoid log(0)

In [ ]:
# Logical operations (the basis of all filtering)
expression <- c(2.5, 3.1, 8.2, 1.8, 5.6)

cat("expression > 3:  ", expression > 3, "\n")
cat("expression >= 5: ", expression >= 5, "\n")

# Combine conditions with & (AND) and | (OR)
in_range <- expression > 2 & expression < 6
cat("2 < expr < 6:    ", in_range, "\n")

# Summary functions on logical vectors
cat("\nHow many > 3?  ", sum(expression > 3), "\n")   # TRUE counts as 1
cat("Any > 10?      ", any(expression > 10), "\n")
cat("All > 0?       ", all(expression > 0), "\n")

In [ ]:
# Useful vector functions
x <- c(4, 1, 7, 2, 9, 3)

cat("length:", length(x), "\n")
cat("sum:   ", sum(x), "\n")
cat("mean:  ", mean(x), "\n")
cat("median:", median(x), "\n")
cat("sd:    ", sd(x), "\n")
cat("min:   ", min(x), "\n")
cat("max:   ", max(x), "\n")
cat("range: ", range(x), "\n")
cat("sort:  ", sort(x), "\n")
cat("order: ", order(x), "\n")  # Indices that would sort the vector

---

## 3. Indexing and Subsetting

R supports three powerful indexing styles:
1. **Positive integers**: Select specific positions
2. **Negative integers**: Exclude specific positions
3. **Logical vectors**: Select where `TRUE`

Remember: **R indexes from 1, not 0!**

In [ ]:
x <- c(10, 20, 30, 40, 50, 60)

# Positive indexing
cat("x[1]       =", x[1], "\n")         # First element: 10
cat("x[2:4]     =", x[2:4], "\n")       # Elements 2 to 4: 20 30 40
cat("x[c(1,3,5)]=", x[c(1, 3, 5)], "\n") # Specific positions: 10 30 50

# Negative indexing = EXCLUDE (very different from Python!)
cat("x[-1]      =", x[-1], "\n")        # All except first: 20 30 40 50 60
cat("x[-(1:3)]  =", x[-(1:3)], "\n")    # Exclude first 3: 40 50 60

# Logical indexing
cat("x[x > 25]  =", x[x > 25], "\n")    # Values > 25: 30 40 50 60

In [ ]:
# Named vectors -- very useful for biological data
gc_content <- c(BRCA1 = 0.42, TP53 = 0.38, EGFR = 0.55, MYC = 0.61)

cat("GC content of EGFR:", gc_content["EGFR"], "\n")
cat("Names:", names(gc_content), "\n")

# Which genes are GC-rich (> 50%)?
gc_rich <- gc_content[gc_content > 0.5]
cat("GC-rich genes:", names(gc_rich), "\n")
cat("Their GC content:", gc_rich, "\n")

### Bioinformatics Example: Filtering Differentially Expressed Genes

In [ ]:
# Simulated differential expression results
genes  <- c("BRCA1", "TP53", "EGFR", "MYC", "KRAS", "PTEN", "RB1", "APC")
log2fc <- c(1.2, -0.5, 3.8, 2.1, -1.8, 0.3, -2.5, 0.1)
pvalue <- c(0.01, 0.15, 0.001, 0.005, 0.02, 0.45, 0.003, 0.72)

# Significantly upregulated: log2FC > 1 AND p < 0.05
sig_up <- log2fc > 1 & pvalue < 0.05
cat("Significantly upregulated:\n")
cat("  Genes:", genes[sig_up], "\n")
cat("  log2FC:", log2fc[sig_up], "\n")

# Significantly downregulated: log2FC < -1 AND p < 0.05
sig_down <- log2fc < -1 & pvalue < 0.05
cat("\nSignificantly downregulated:\n")
cat("  Genes:", genes[sig_down], "\n")
cat("  log2FC:", log2fc[sig_down], "\n")

# which() returns the indices of TRUE values
cat("\nIndices of significant genes:", which(sig_up | sig_down), "\n")

---

## 4. Matrices

Matrices are 2D arrays where all elements have the same type. They are used for expression matrices, distance matrices, and substitution matrices.

In [ ]:
# Create an expression matrix (genes x samples)
expr_matrix <- matrix(
    c(5.2, 3.1, 8.5, 6.2,
      4.8, 2.9, 7.1, 5.8,
      6.1, 3.5, 9.2, 7.0),
    nrow = 3, byrow = TRUE,
    dimnames = list(
        c("Sample1", "Sample2", "Sample3"),
        c("BRCA1", "TP53", "EGFR", "MYC")
    )
)

print(expr_matrix)
cat("\nDimensions:", dim(expr_matrix), "\n")
cat("Row names:", rownames(expr_matrix), "\n")
cat("Col names:", colnames(expr_matrix), "\n")

In [ ]:
# Matrix indexing: [row, column]
cat("EGFR in Sample2:", expr_matrix[2, "EGFR"], "\n")
cat("All of Sample1: ", expr_matrix[1, ], "\n")
cat("All BRCA1 values:", expr_matrix[, "BRCA1"], "\n")

# Apply functions across rows or columns
cat("\nMean per sample (rows): ", apply(expr_matrix, 1, mean), "\n")
cat("Mean per gene (columns):", apply(expr_matrix, 2, mean), "\n")
cat("SD per gene:            ", round(apply(expr_matrix, 2, sd), 2), "\n")

---

## 5. Data Frames -- Tabular Data

Data frames are R's equivalent of a spreadsheet or a Pandas DataFrame. Each column can have a different type. This is the primary data structure for biological datasets.

In [ ]:
# Create a data frame
gene_data <- data.frame(
    gene = c("BRCA1", "TP53", "EGFR", "MYC", "KRAS", "PTEN"),
    expression = c(5.2, 3.1, 8.5, 6.2, 4.1, 2.8),
    log2fc = c(1.2, -0.5, 3.8, 2.1, -1.8, -2.1),
    pvalue = c(0.01, 0.15, 0.001, 0.005, 0.02, 0.008),
    is_oncogene = c(FALSE, FALSE, TRUE, TRUE, TRUE, FALSE),
    stringsAsFactors = FALSE  # Keep strings as character, not factor
)

print(gene_data)

In [ ]:
# Inspecting data frames
cat("Dimensions:", dim(gene_data), "\n")
cat("Column names:", colnames(gene_data), "\n")
str(gene_data)   # Structure -- very useful
cat("\n")
summary(gene_data)  # Statistical summary of each column

In [ ]:
# Accessing data
cat("Gene column:", gene_data$gene, "\n")
cat("First row:\n")
print(gene_data[1, ])
cat("\nRow 2, column 'log2fc':", gene_data[2, "log2fc"], "\n")

# Filtering -- the most common operation
cat("\nOncogenes:\n")
print(gene_data[gene_data$is_oncogene, ])

cat("\nSignificantly upregulated (log2FC > 1, p < 0.05):\n")
sig_up <- gene_data[gene_data$log2fc > 1 & gene_data$pvalue < 0.05, ]
print(sig_up)

In [ ]:
# Modifying data frames

# Add a new column
gene_data$significant <- gene_data$pvalue < 0.05

# Bonferroni correction
gene_data$padj <- p.adjust(gene_data$pvalue, method = "bonferroni")

# BH/FDR correction (standard in genomics)
gene_data$fdr <- p.adjust(gene_data$pvalue, method = "BH")

# Sorting
gene_data_sorted <- gene_data[order(gene_data$pvalue), ]
print(gene_data_sorted)

---

## 6. Statistical Distributions

R has a consistent naming convention for distribution functions:

| Prefix | Meaning | Example |
|--------|---------|--------|
| `d` | Density (PDF value) | `dnorm(x)` |
| `p` | Cumulative probability (CDF) | `pnorm(q)` |
| `q` | Quantile (inverse CDF) | `qnorm(p)` |
| `r` | Random sampling | `rnorm(n)` |

Common distributions: `norm` (Normal), `t` (Student's t), `binom` (Binomial), `pois` (Poisson), `nbinom` (Negative binomial), `unif` (Uniform), `chisq` (Chi-squared), `f` (F).

In [ ]:
# Normal distribution
set.seed(42)  # For reproducibility

# Generate random expression values
expression_values <- rnorm(n = 100, mean = 10, sd = 2)

cat("Summary of simulated expression data:\n")
cat("  Mean:  ", round(mean(expression_values), 2), "\n")
cat("  SD:    ", round(sd(expression_values), 2), "\n")
cat("  Min:   ", round(min(expression_values), 2), "\n")
cat("  Max:   ", round(max(expression_values), 2), "\n")
cat("\n")
summary(expression_values)

In [ ]:
# Probability calculations

# What fraction of gene expression values fall below 8? (mean=10, sd=2)
cat("P(X < 8):  ", round(pnorm(8, mean = 10, sd = 2), 4), "\n")

# What expression level corresponds to the 95th percentile?
cat("95th percentile:", round(qnorm(0.95, mean = 10, sd = 2), 2), "\n")

# Density at the mean
cat("Density at x=10:", round(dnorm(10, mean = 10, sd = 2), 4), "\n")

In [ ]:
# Distributions commonly used in bioinformatics
set.seed(42)

# Uniform: random sampling, permutation tests
cat("Uniform(0,1):", round(runif(5, min = 0, max = 1), 3), "\n")

# Binomial: variant calling (success/failure per read)
cat("Binomial(n=100, p=0.5):", rbinom(5, size = 100, prob = 0.5), "\n")

# Poisson: read counts (simple model)
cat("Poisson(lambda=20):", rpois(5, lambda = 20), "\n")

# Negative binomial: RNA-seq counts (accounts for overdispersion)
# This is what DESeq2 uses!
cat("NegBinom(size=5, mu=20):", rnbinom(5, size = 5, mu = 20), "\n")

---

## 7. Basic Plotting

R's base graphics are quick and effective for data exploration. For publication-quality figures, most bioinformaticians use **ggplot2**, but base R plots are excellent for rapid inspection.

In [ ]:
# Histogram of expression values
set.seed(42)
expression <- rnorm(500, mean = 10, sd = 2)

hist(expression,
     breaks = 30,
     main = "Simulated Gene Expression Distribution",
     xlab = "Expression Level (log2 TPM)",
     ylab = "Frequency",
     col = "steelblue",
     border = "white")

# Add mean line
abline(v = mean(expression), col = "red", lwd = 2, lty = 2)

In [ ]:
# Boxplot: comparing groups
set.seed(42)
control <- rnorm(50, mean = 10, sd = 2)
treatment_A <- rnorm(50, mean = 13, sd = 2)
treatment_B <- rnorm(50, mean = 11, sd = 3)

boxplot(
    list(Control = control, "Treatment A" = treatment_A, "Treatment B" = treatment_B),
    main = "Gene Expression by Treatment Group",
    ylab = "Expression Level",
    col = c("lightblue", "lightcoral", "lightgreen"),
    las = 1  # Horizontal y-axis labels
)

In [ ]:
# Scatter plot: MA plot (common in differential expression analysis)
set.seed(42)
n_genes <- 200
mean_expr <- rnorm(n_genes, mean = 8, sd = 3)  # Average expression (A)
log2fc <- rnorm(n_genes, mean = 0, sd = 0.5)    # Log2 fold change (M)

# Add some truly differentially expressed genes
n_de <- 20
mean_expr <- c(mean_expr, rnorm(n_de, mean = 10, sd = 2))
log2fc <- c(log2fc, rnorm(n_de, mean = 2.5, sd = 0.5))

# Color based on significance
is_sig <- abs(log2fc) > 1
colors <- ifelse(is_sig, "red", "gray60")

plot(mean_expr, log2fc,
     pch = 16, cex = 0.7,
     col = colors,
     main = "MA Plot (Differential Expression)",
     xlab = "Mean Expression (log2)",
     ylab = "Log2 Fold Change")
abline(h = c(-1, 0, 1), lty = c(2, 1, 2), col = c("blue", "black", "blue"))
legend("topright", legend = c("Significant", "Not significant"),
       col = c("red", "gray60"), pch = 16)

In [ ]:
# Volcano plot -- the classic differential expression visualization
set.seed(42)
n <- 500
log2fc_all <- rnorm(n, 0, 0.8)
pvals_all <- 10^(runif(n, -0.3, 0))

# Add DE genes
n_de <- 30
log2fc_de <- c(rnorm(n_de/2, 2.5, 0.5), rnorm(n_de/2, -2.5, 0.5))
pvals_de <- 10^(runif(n_de, -6, -2))

all_log2fc <- c(log2fc_all, log2fc_de)
all_pvals <- c(pvals_all, pvals_de)
neg_log10p <- -log10(all_pvals)

sig <- abs(all_log2fc) > 1 & all_pvals < 0.05
colors <- ifelse(sig & all_log2fc > 0, "firebrick",
          ifelse(sig & all_log2fc < 0, "steelblue", "gray70"))

plot(all_log2fc, neg_log10p,
     pch = 16, cex = 0.6, col = colors,
     main = "Volcano Plot",
     xlab = "Log2 Fold Change",
     ylab = "-Log10(p-value)")
abline(v = c(-1, 1), lty = 2, col = "gray40")
abline(h = -log10(0.05), lty = 2, col = "gray40")
legend("topright", legend = c("Up", "Down", "NS"),
       col = c("firebrick", "steelblue", "gray70"), pch = 16, cex = 0.8)

---

## 8. Reading and Writing Files

R has built-in functions for tabular data. Bioinformatics files are often tab-delimited with headers.

In [ ]:
# Reading files (commented out -- adapt paths to your system)

# CSV file
# data <- read.csv("expression_data.csv")

# Tab-delimited file (very common in bioinformatics: BED, GFF, count matrices)
# data <- read.table("counts.tsv", header = TRUE, sep = "\t")

# With row names (e.g., gene names as row identifiers)
# data <- read.csv("data.csv", row.names = 1)

# Writing files
# write.csv(gene_data, "output.csv", row.names = FALSE)
# write.table(gene_data, "output.tsv", sep = "\t", quote = FALSE, row.names = FALSE)

# Quick demo with a temp file
tmpfile <- tempfile(fileext = ".csv")
write.csv(gene_data, tmpfile, row.names = FALSE)

# Read it back
reloaded <- read.csv(tmpfile)
cat("Wrote and read back", nrow(reloaded), "rows and", ncol(reloaded), "columns\n")
head(reloaded, 3)

---

## 9. Basic Statistical Tests in R

R was built for statistics. Common tests are one function call away.

In [ ]:
# t-test: compare gene expression between two groups
set.seed(42)
control <- rnorm(30, mean = 10, sd = 2)
treatment <- rnorm(30, mean = 12, sd = 2)

result <- t.test(control, treatment)
print(result)

cat("\nInterpretation:\n")
cat("  p-value:", format(result$p.value, digits = 4), "\n")
cat("  Conclusion:", ifelse(result$p.value < 0.05, "Significant difference", "No significant difference"), "\n")

In [ ]:
# Non-parametric alternative: Wilcoxon rank-sum test (Mann-Whitney U)
wilcox_result <- wilcox.test(control, treatment)
cat("Wilcoxon p-value:", format(wilcox_result$p.value, digits = 4), "\n")

# Normality test
shapiro_result <- shapiro.test(control)
cat("\nShapiro-Wilk test for normality (control group):\n")
cat("  p-value:", format(shapiro_result$p.value, digits = 4), "\n")
cat("  Normal?", ifelse(shapiro_result$p.value > 0.05, "Yes (fail to reject H0)", "No (reject H0)"), "\n")

In [ ]:
# Correlation
set.seed(42)
gene_a <- rnorm(50, 10, 2)
gene_b <- gene_a * 0.8 + rnorm(50, 0, 1.5)  # Correlated with noise

# Pearson correlation (linear relationship, assumes normality)
pearson <- cor.test(gene_a, gene_b, method = "pearson")
cat("Pearson r:", round(pearson$estimate, 3), " p:", format(pearson$p.value, digits = 3), "\n")

# Spearman correlation (rank-based, no normality assumption)
spearman <- cor.test(gene_a, gene_b, method = "spearman")
cat("Spearman rho:", round(spearman$estimate, 3), " p:", format(spearman$p.value, digits = 3), "\n")

# Plot
plot(gene_a, gene_b, pch = 16, col = "steelblue",
     main = paste0("Gene Co-expression (r = ", round(pearson$estimate, 2), ")"),
     xlab = "Gene A expression", ylab = "Gene B expression")
abline(lm(gene_b ~ gene_a), col = "red", lwd = 2)

---

## 10. Introduction to Bioconductor

**Bioconductor** (https://bioconductor.org) is R's ecosystem for genomic data analysis. It provides:

| Category | Key Packages |
|----------|-------------|
| Differential expression | DESeq2, edgeR, limma |
| Genomic ranges | GenomicRanges, IRanges |
| Sequence analysis | Biostrings, BSgenome |
| Annotation | AnnotationDbi, org.Hs.eg.db |
| Single-cell | SingleCellExperiment, scran |
| Visualization | ComplexHeatmap, EnhancedVolcano |
| Workflows | Rsamtools, GenomicAlignments |

### Installing Bioconductor

```r
# Install BiocManager (only once)
install.packages("BiocManager")

# Install Bioconductor packages
BiocManager::install("DESeq2")
BiocManager::install("GenomicRanges")
BiocManager::install("Biostrings")
```

### A Taste of DESeq2

DESeq2 is the gold standard for RNA-seq differential expression analysis. Here is the typical workflow (conceptual -- requires actual count data):

```r
library(DESeq2)

# 1. Create DESeq dataset from count matrix
dds <- DESeqDataSetFromMatrix(
    countData = count_matrix,       # Genes x Samples (raw counts)
    colData = sample_info,          # Sample metadata
    design = ~ condition            # What to compare
)

# 2. Run the analysis (normalization + dispersion estimation + testing)
dds <- DESeq(dds)

# 3. Extract results
results <- results(dds, contrast = c("condition", "treatment", "control"))

# 4. Filter significant genes
sig_genes <- results[which(results$padj < 0.05 & abs(results$log2FoldChange) > 1), ]
```

In [ ]:
# Simulating what DESeq2 output looks like
# (so you know how to read it even without installing DESeq2)
set.seed(42)
n_genes <- 100

deseq_results <- data.frame(
    gene = paste0("Gene_", 1:n_genes),
    baseMean = abs(rnorm(n_genes, 500, 300)),
    log2FoldChange = rnorm(n_genes, 0, 1),
    lfcSE = abs(rnorm(n_genes, 0.3, 0.1)),
    pvalue = 10^(runif(n_genes, -5, 0)),
    stringsAsFactors = FALSE
)

# Add FDR-adjusted p-values (this is what DESeq2 does automatically)
deseq_results$padj <- p.adjust(deseq_results$pvalue, method = "BH")

# Significant genes
sig <- deseq_results[deseq_results$padj < 0.05 & abs(deseq_results$log2FoldChange) > 1, ]
cat("Significant DE genes (|log2FC| > 1, FDR < 0.05):", nrow(sig), "out of", n_genes, "\n\n")

# Show top results sorted by adjusted p-value
top_results <- head(deseq_results[order(deseq_results$padj), ], 10)
print(top_results)

---

## 11. Quick Reference

### Vectors
```r
x <- c(1, 2, 3)       # Create vector
x <- 1:10             # Integer sequence
x[1]                  # First element (indexing from 1!)
x[-1]                 # All EXCEPT first
x[x > 5]              # Filter by condition
length(x)             # Number of elements
```

### Data Frames
```r
df <- data.frame(a = 1:3, b = c("x", "y", "z"))
df$a                  # Access column by name
df[1, ]               # First row
df[df$a > 1, ]        # Filter rows
dim(df)               # Rows, columns
str(df)               # Structure
```

### Statistics
```r
mean(x); median(x); sd(x)           # Descriptive stats
t.test(x, y)                         # Two-sample t-test
wilcox.test(x, y)                    # Non-parametric alternative
cor.test(x, y)                       # Correlation test
p.adjust(pvals, method = "BH")       # Multiple testing correction
```

### Plotting
```r
plot(x, y)            # Scatter plot
hist(x, breaks = 30)  # Histogram
boxplot(x ~ group)    # Boxplot by group
abline(h = 0)         # Add horizontal line
```

### Key Takeaways

1. **R indexes from 1**, not 0 -- the most common source of bugs for Python users
2. **Vectorized operations** eliminate the need for most loops
3. **Negative indexing excludes** elements (opposite of Python)
4. **Data frames** are R's workhorse for tabular biological data
5. **Distribution functions** follow the `d/p/q/r` prefix pattern
6. **Bioconductor** is essential for genomics -- DESeq2, edgeR, GenomicRanges

---

## Exercises

### Exercise 1: Vector Manipulation

Given the following gene expression data:
```r
genes <- c("GAPDH", "BRCA1", "TP53", "EGFR", "MYC", "KRAS", "PTEN", "RB1", "APC", "CDH1")
expr <- c(15.2, 4.8, 3.2, 9.1, 7.5, 5.3, 2.1, 3.8, 1.9, 6.4)
```

1. Find the mean and standard deviation of expression values
2. Which genes have expression above the mean?
3. Sort the genes by expression (highest first)
4. What is the expression of the 3rd highest-expressed gene?

In [ ]:
# YOUR CODE HERE


### Exercise 2: Data Frame Analysis

Create a data frame with the following columns for 8 genes:
- `gene`: gene name
- `log2fc`: log2 fold change (use any realistic values)
- `pvalue`: p-value
- `chromosome`: chromosome number

Then:
1. Add an `fdr` column using `p.adjust()` with BH method
2. Filter for significantly differentially expressed genes (FDR < 0.1, |log2FC| > 1)
3. Sort by absolute fold change (largest first)

In [ ]:
# YOUR CODE HERE


### Exercise 3: Plotting

Generate two groups of data:
- `healthy`: 40 samples from N(mean=5, sd=1.5)
- `tumor`: 40 samples from N(mean=8, sd=2)

1. Create a boxplot comparing the two groups
2. Run a t-test and display the p-value in the plot title
3. Create a histogram of each group overlaid on the same plot (use `add = TRUE` for the second `hist()`)

In [ ]:
# YOUR CODE HERE


---

## Exercise Solutions

In [ ]:
# Solution 1: Vector Manipulation
genes <- c("GAPDH", "BRCA1", "TP53", "EGFR", "MYC", "KRAS", "PTEN", "RB1", "APC", "CDH1")
expr <- c(15.2, 4.8, 3.2, 9.1, 7.5, 5.3, 2.1, 3.8, 1.9, 6.4)

# 1. Mean and SD
cat("Mean:", mean(expr), "\n")
cat("SD:", sd(expr), "\n\n")

# 2. Genes above mean
above_mean <- genes[expr > mean(expr)]
cat("Above mean:", above_mean, "\n\n")

# 3. Sort by expression (highest first)
sorted_idx <- order(expr, decreasing = TRUE)
cat("Sorted genes:", genes[sorted_idx], "\n")
cat("Sorted expr: ", expr[sorted_idx], "\n\n")

# 4. 3rd highest
cat("3rd highest:", genes[sorted_idx[3]], "with expression", expr[sorted_idx[3]], "\n")

In [ ]:
# Solution 2: Data Frame Analysis
de_data <- data.frame(
    gene = c("BRCA1", "TP53", "EGFR", "MYC", "KRAS", "PTEN", "RB1", "CDH1"),
    log2fc = c(1.5, -0.3, 3.2, 2.8, -1.9, -2.4, 0.7, 1.1),
    pvalue = c(0.002, 0.35, 0.0001, 0.001, 0.008, 0.003, 0.12, 0.04),
    chromosome = c(17, 17, 7, 8, 12, 10, 13, 16),
    stringsAsFactors = FALSE
)

# 1. Add FDR
de_data$fdr <- p.adjust(de_data$pvalue, method = "BH")

# 2. Filter significant DE genes
sig_de <- de_data[de_data$fdr < 0.1 & abs(de_data$log2fc) > 1, ]
cat("Significant DE genes:\n")
print(sig_de)

# 3. Sort by absolute fold change
sig_de_sorted <- sig_de[order(abs(sig_de$log2fc), decreasing = TRUE), ]
cat("\nSorted by |log2FC|:\n")
print(sig_de_sorted)

In [ ]:
# Solution 3: Plotting
set.seed(42)
healthy <- rnorm(40, mean = 5, sd = 1.5)
tumor <- rnorm(40, mean = 8, sd = 2)

# t-test
tt <- t.test(healthy, tumor)

# Boxplot with p-value in title
boxplot(list(Healthy = healthy, Tumor = tumor),
        main = paste0("Expression Comparison (p = ", format(tt$p.value, digits = 3), ")"),
        ylab = "Expression Level",
        col = c("lightgreen", "lightcoral"))

In [ ]:
# Overlaid histograms
hist(healthy, breaks = 15, col = rgb(0, 0.6, 0, 0.4), xlim = c(0, 15),
     main = "Expression Distribution: Healthy vs Tumor",
     xlab = "Expression Level")
hist(tumor, breaks = 15, col = rgb(0.8, 0, 0, 0.4), add = TRUE)
legend("topright", legend = c("Healthy", "Tumor"),
       fill = c(rgb(0, 0.6, 0, 0.4), rgb(0.8, 0, 0, 0.4)))

---

## 12. Categorical Data Analysis

In biology, many variables are categorical: treatment groups, mutation status (mutated/wildtype), phenotype categories, or tissue types. R's `table()` function builds contingency tables from categorical vectors, and a family of tests — binomial, chi-square, and Fisher's exact — let you ask whether observed counts deviate from expectation or whether two categorical variables are associated.

In [ ]:
# Bioinformatics example: mutation status across treatment groups
set.seed(42)

# Simulate 80 patients: treatment group and KRAS mutation status
treatment <- factor(rep(c("Control", "DrugA", "DrugB"), c(30, 25, 25)))
mutation  <- factor(
    c(
        sample(c("Mutated", "Wildtype"), 30, replace = TRUE, prob = c(0.3, 0.7)),
        sample(c("Mutated", "Wildtype"), 25, replace = TRUE, prob = c(0.6, 0.4)),
        sample(c("Mutated", "Wildtype"), 25, replace = TRUE, prob = c(0.5, 0.5))
    )
)

# 1D table: overall mutation frequency
t1 <- table(mutation)
t1

# 2D contingency table: mutation status by treatment group
t2 <- table(mutation = mutation, treatment = treatment)
t2

# Row proportions: what fraction of each mutation status is in each group
prop.table(t2, margin = 1)

In [ ]:
# Visualise the contingency table
barplot(t2,
        beside    = TRUE,
        legend    = TRUE,
        args.legend = list(x = "topright"),
        col       = c("tomato", "steelblue"),
        main      = "KRAS Mutation Status by Treatment Group",
        xlab      = "Treatment",
        ylab      = "Count")

mosaicplot(t2,
           main  = "Mosaic plot: Mutation x Treatment",
           color = c("tomato", "steelblue"))

In [ ]:
# Binomial test: is the overall mutation rate different from 50 %?
binom.test(t1)          # tests whether Mutated == Wildtype

# Chi-square test: is mutation status associated with treatment group?
chi <- chisq.test(t2)
chi

# Inspect expected counts (assumption: all > 5 for chi-square to be valid)
chi$expected

# Fisher's exact test: appropriate for small samples or low expected counts
fisher.test(t2)

---

## 13. Hypothesis Testing in R

A standard hypothesis-testing workflow in R follows three steps:

1. **Check normality** — `shapiro.test()` and a Q-Q plot tell you whether your data are approximately normal.
2. **Check variance equality** — `bartlett.test()` (or `var.test()`) checks the equal-variance assumption.
3. **Choose the right test** — use a parametric test (t-test) when normality holds, otherwise use a non-parametric alternative (Wilcoxon rank-sum).

Both tests answer the same question: "Do these two groups come from the same distribution?" They differ in what they assume about that distribution.

In [ ]:
# Simulate gene expression in tumor vs. normal samples
set.seed(42)
normal_expr <- rnorm(30, mean = 6.5, sd = 1.2)   # Normal tissue
tumor_expr  <- rnorm(30, mean = 8.8, sd = 1.9)   # Tumor tissue

# Step 1: Check normality with Shapiro-Wilk (H0: data are normal)
shapiro.test(normal_expr)
shapiro.test(tumor_expr)

# Visual check: Q-Q plot
par(mfrow = c(1, 2))
qqnorm(normal_expr, main = "Normal tissue"); qqline(normal_expr, col = "steelblue")
qqnorm(tumor_expr,  main = "Tumor tissue");  qqline(tumor_expr,  col = "tomato")
par(mfrow = c(1, 1))

In [ ]:
# Step 2: Check variance equality (Bartlett's test)
# Combine into a data frame for formula-style tests
expr   <- c(normal_expr, tumor_expr)
tissue <- factor(rep(c("Normal", "Tumor"), each = 30))

bartlett.test(expr ~ tissue)   # H0: variances are equal

# Step 3a: Parametric -- Welch t-test (does not assume equal variances)
t_result <- t.test(expr ~ tissue)
t_result

cat("p-value:", t_result$p.value, "\n")
cat("95% CI: [", t_result$conf.int[1], ",", t_result$conf.int[2], "]\n")

In [ ]:
# Step 3b: Non-parametric alternative -- Wilcoxon rank-sum test
# Use when Shapiro-Wilk p < 0.05 (normality violated)
wilcox_result <- wilcox.test(expr ~ tissue)
wilcox_result

# Rule of thumb:
#   Shapiro p > 0.05  -->  use t.test()
#   Shapiro p < 0.05  -->  use wilcox.test()
#   For n > 30, the t-test is fairly robust to non-normality (CLT)

cat("\nSummary:\n")
cat("t-test p-value:      ", t_result$p.value, "\n")
cat("Wilcoxon p-value:    ", wilcox_result$p.value, "\n")

---

## 14. One-Way ANOVA

When you need to compare means across three or more groups (e.g., gene expression in three tissue types), a t-test is no longer appropriate — running multiple t-tests inflates the false-positive rate. ANOVA tests the null hypothesis that all group means are equal, and post-hoc tests (Tukey HSD) identify which specific pairs differ.

In [ ]:
# Simulate EGFR expression across three tissue types
set.seed(42)
normal  <- rnorm(25, mean = 5.0, sd = 1.0)
primary <- rnorm(25, mean = 7.5, sd = 1.3)
meta    <- rnorm(25, mean = 9.8, sd = 1.5)

expr_df <- data.frame(
    expression = c(normal, primary, meta),
    tissue     = factor(rep(c("Normal", "Primary", "Metastatic"), each = 25))
)

# Boxplot to visualise group differences
boxplot(expression ~ tissue, data = expr_df,
        main = "EGFR Expression by Tissue Type",
        ylab = "log2 expression",
        col  = c("lightgreen", "lightyellow", "tomato"))

In [ ]:
# One-way ANOVA
fit <- aov(expression ~ tissue, data = expr_df)
summary(fit)

# A significant F-test means at least one group mean differs --
# but not which pairs. Use Tukey HSD for pairwise comparisons.
TukeyHSD(fit)

---

## 15. Correlation Analysis

Correlation measures the strength of association between continuous variables. In bioinformatics, you'll test correlations between gene expression levels, between drug dose and response, or between phylogenetic distance and sequence divergence.

In [ ]:
# Pearson correlation between two gene expression profiles
set.seed(42)
gene_a <- rnorm(50, mean = 10, sd = 2)          # Gene A expression
gene_b <- gene_a * 0.75 + rnorm(50, 0, 1.5)    # Gene B: correlated with A

result <- cor.test(gene_a, gene_b, method = "pearson")
cat("Pearson r  :", round(result$estimate, 4), "\n")
cat("p-value    :", signif(result$p.value, 4),  "\n")
cat("95% CI     :", round(result$conf.int, 4),  "\n")

In [ ]:
# Correlation matrix across multiple genes
set.seed(42)
n <- 60
gene_data <- data.frame(
    BRCA1 = rnorm(n, 10, 2),
    TP53  = rnorm(n, 8,  2),
    EGFR  = rnorm(n, 6,  1.5),
    MYC   = rnorm(n, 12, 3)
)
gene_data$MDM2 <- gene_data$TP53 * 0.6 + rnorm(n, 0, 1)   # MDM2 tracks TP53

cat("Correlation matrix:\n")
print(round(cor(gene_data), 3))

# Visual pairwise scatter plots
pairs(gene_data, main = "Pairwise gene expression correlations",
      col = "steelblue", pch = 16, cex = 0.7)

In [ ]:
# Scatter plot with regression line: TP53 vs MDM2
plot(gene_data$TP53, gene_data$MDM2,
     xlab = "TP53 expression",
     ylab = "MDM2 expression",
     main = "TP53 vs MDM2 (expected co-regulation)",
     pch  = 16, col = "steelblue")
abline(lm(MDM2 ~ TP53, data = gene_data), col = "firebrick", lwd = 2)

tp53_mdm2 <- cor.test(gene_data$TP53, gene_data$MDM2)
legend("topleft",
       legend = paste0("r = ", round(tp53_mdm2$estimate, 3),
                       ",  p = ", signif(tp53_mdm2$p.value, 3)),
       bty = "n")

---

## 16. Phylogenetic Trees in R

R has powerful packages for phylogenetic tree visualization. The phytools and ape packages can read Newick format trees and produce publication-quality phylograms.

> **Connection:** This section links directly to **Tier 2 Module 06 (Phylogenetics)**, where you will build trees from real sequence alignments.

In [ ]:
# Basic phylogenetic tree with the ape package
# install.packages("ape")   # run once if needed
library(ape)

# A small Newick tree string (could also come from read.tree(file = "..."))
newick_str <- "((Human:0.05, Chimp:0.04):0.10, (Mouse:0.15, Rat:0.14):0.12, Zebrafish:0.35);"
tree <- read.tree(text = newick_str)

# Basic phylogram
plot(tree,
     type  = "phylogram",
     main  = "Primate/Rodent example tree",
     cex   = 1.0,
     edge.width = 2,
     edge.color = "steelblue")
axisPhylo()   # add a branch-length axis

---

## 17. Data Wrangling with dplyr

The tidyverse (dplyr, ggplot2) is the modern R ecosystem for data manipulation. Many bioinformatics workflows in R use these tools.

In [ ]:
# install.packages(c("dplyr", "ggplot2"))   # run once if needed
library(dplyr)
library(ggplot2)

# Simulate a gene-expression data frame
set.seed(42)
tissues <- c("liver", "brain", "kidney")
expr_df <- data.frame(
    gene       = rep(c("BRCA1","TP53","EGFR","MYC"), each = 30),
    tissue     = rep(rep(tissues, each = 10), times = 4),
    expression = c(
        rnorm(30, mean = 8,  sd = 1.5),
        rnorm(30, mean = 10, sd = 2.0),
        rnorm(30, mean = 6,  sd = 1.2),
        rnorm(30, mean = 12, sd = 2.5)
    ),
    logFC      = rnorm(120, 0, 1.5)
)

# --- dplyr verbs ---
# filter: keep only EGFR rows from liver or brain
egfr_sub <- expr_df %>%
    filter(gene == "EGFR", tissue %in% c("liver", "brain"))

# select: keep only relevant columns
egfr_sub <- egfr_sub %>% select(gene, tissue, expression)

# mutate: add a z-score column
egfr_sub <- egfr_sub %>%
    mutate(z_score = (expression - mean(expression)) / sd(expression))

# summarize + group_by: mean expression per tissue per gene
summary_df <- expr_df %>%
    group_by(gene, tissue) %>%
    summarise(mean_expr = mean(expression),
              sd_expr   = sd(expression),
              .groups   = "drop")

cat("Mean expression per gene per tissue:\n")
print(summary_df)

In [ ]:
# ggplot2: boxplot of expression by tissue, faceted by gene
ggplot(expr_df, aes(x = tissue, y = expression, fill = tissue)) +
    geom_boxplot(alpha = 0.7, outlier.size = 1) +
    facet_wrap(~ gene, scales = "free_y") +
    labs(title  = "Gene expression by tissue type",
         x      = "Tissue",
         y      = "Normalised expression",
         fill   = "Tissue") +
    theme_bw() +
    theme(legend.position = "none",
          strip.text = element_text(face = "bold"))

---

[< Previous: Character Encodings and Binary Data in Bioinfor...](../04_File_Encodings/01_character_encodings.ipynb) | [Tier 0: Computational Foundations Overview](../README.md) | [Next: Biostatistics Fundamentals >](../06_Biostatistics/01_biostatistics_fundamentals.ipynb)

---